In [1]:
# ==========================================================
# PART 1: SETUP & LIBRARY INSTALLATION
# ==========================================================

# Install libraries (Run once in Google Colab)
!pip install -q yfinance fredapi ta torch scikit-learn matplotlib

# -------------------------
# Import Libraries
# -------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from fredapi import Fred

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import ta

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# -------------------------
# Set Random Seed
# -------------------------
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# -------------------------
# Check Device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print("Device:", device)
print("=" * 50)

# -------------------------
# Plot Settings
# -------------------------
plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)

print("Libraries Loaded Successfully!")

  Preparing metadata (setup.py) ... done
Device: cuda
Libraries Loaded Successfully!


In [3]:
# ==========================================================
# PART 2: DOWNLOAD MARKET DATA
# ==========================================================

# -------------------------
# Date Range
# -------------------------
start_date = "2015-01-01"
end_date = "2025-12-31"

# -------------------------
# Download Data
# -------------------------
gold = yf.download("GC=F", start=start_date, end=end_date)
sp500 = yf.download("^GSPC", start=start_date, end=end_date)
vix = yf.download("^VIX", start=start_date, end=end_date)
dxy = yf.download("DX-Y.NYB", start=start_date, end=end_date)
yield10 = yf.download("^TNX", start=start_date, end=end_date)

# -------------------------
# Create DataFrame
# -------------------------
df = pd.DataFrame(index=gold.index)

df["Gold"] = gold["Close"]
df["SP500"] = sp500["Close"]
df["VIX"] = vix["Close"]
df["Dollar_Index"] = dxy["Close"]
df["US10Y"] = yield10["Close"]

# -------------------------
# Handle Missing Values
# -------------------------
df = df.ffill().bfill()
df.dropna(inplace=True)

# -------------------------
# Display Information
# -------------------------
print("=" * 60)
print("Dataset Shape:", df.shape)
print("=" * 60)

print(df.head())

print("\nColumns:")
print(df.columns.tolist())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Dataset Shape: (2764, 5)
                   Gold        SP500        VIX  Dollar_Index  US10Y
Date                                                                
2015-01-02  1186.000000  2058.199951  17.790001     91.080002  2.123
2015-01-05  1203.900024  2020.579956  19.920000     91.379997  2.039
2015-01-06  1219.300049  2002.609985  21.120001     91.500000  1.963
2015-01-07  1210.599976  2025.900024  19.309999     91.889999  1.954
2015-01-08  1208.400024  2062.139893  17.010000     92.370003  2.016

Columns:
['Gold', 'SP500', 'VIX', 'Dollar_Index', 'US10Y']


In [4]:
# ==========================================================
# PART 3: FEATURE ENGINEERING
# ==========================================================

# -------------------------
# Daily Returns
# -------------------------
df["Gold_Return"] = df["Gold"].pct_change()
df["SP500_Return"] = df["SP500"].pct_change()
df["Dollar_Return"] = df["Dollar_Index"].pct_change()

# -------------------------
# Moving Averages
# -------------------------
df["MA20"] = df["Gold"].rolling(window=20).mean()
df["MA50"] = df["Gold"].rolling(window=50).mean()

# -------------------------
# Rolling Volatility
# -------------------------
df["Volatility_20"] = df["Gold_Return"].rolling(window=20).std()

# -------------------------
# RSI
# -------------------------
df["RSI"] = ta.momentum.RSIIndicator(
    close=df["Gold"],
    window=14
).rsi()

# -------------------------
# MACD
# -------------------------
macd = ta.trend.MACD(df["Gold"])

df["MACD"] = macd.macd()
df["MACD_Signal"] = macd.macd_signal()

# -------------------------
# Remove Missing Values
# -------------------------
df.dropna(inplace=True)

# -------------------------
# Dataset Information
# -------------------------
print("="*60)
print("Dataset Shape:", df.shape)
print("="*60)

print(df.head())

print("\nFeature Columns:")
print(df.columns.tolist())

Dataset Shape: (2715, 14)
                   Gold        SP500    VIX  Dollar_Index  US10Y  Gold_Return  \
Date                                                                            
2015-03-16  1153.300049  2081.189941  15.61     99.599998  2.098     0.000607   
2015-03-17  1148.300049  2074.280029  15.66     99.589996  2.058    -0.004335   
2015-03-18  1151.400024  2099.500000  13.97     98.550003  1.951     0.002700   
2015-03-19  1169.099976  2089.270020  14.07     99.099998  1.977     0.015373   
2015-03-20  1184.800049  2108.100098  13.02     97.910004  1.930     0.013429   

            SP500_Return  Dollar_Return         MA20         MA50  \
Date                                                                
2015-03-16      0.013534      -0.007276  1187.359991  1224.785994   
2015-03-17     -0.003320      -0.000100  1184.369995  1224.031995   
2015-03-18      0.012158      -0.010443  1181.954999  1222.981995   
2015-03-19     -0.004873       0.005581  1180.054999  1221.97

In [13]:
# ==========================================================
# PART 4: DATA PREPARATION
# ==========================================================

from sklearn.preprocessing import MinMaxScaler
import numpy as np

# -------------------------
# Select Features
# -------------------------
features = [
    "Gold",
    "SP500",
    "VIX",
    "Dollar_Index",
    "US10Y",
    "Gold_Return",
    "SP500_Return",
    "Dollar_Return",
    "MA20",
    "MA50",
    "Volatility_20",
    "RSI",
    "MACD",
    "MACD_Signal"
]

# -------------------------
# Scale Features
# -------------------------
scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(df[features])

# -------------------------
# Sequence Parameters
# -------------------------
lookback = 90

X = []
y = []

# -------------------------
# Create Sequences
# -------------------------
for i in range(lookback, len(scaled_data)):

    # Previous 90 days
    X.append(scaled_data[i-lookback:i])

    # Next day's Gold price (Target)
    y.append(scaled_data[i, 0])

# Convert to NumPy arrays
X = np.array(X)
y = np.array(y)

print("=" * 50)
print("Input Shape :", X.shape)
print("Target Shape:", y.shape)
print("=" * 50)

# Expected Output
# X Shape: (samples, 90, 14)
# y Shape: (samples,)

Input Shape : (2625, 90, 14)
Target Shape: (2625,)


In [18]:
# ==========================================================
# PART 5: TRAIN-TEST SPLIT & PYTORCH DATALOADER
# ==========================================================

from sklearn.model_selection import TimeSeriesSplit
import torch
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# Time Series Split
# -------------------------
tscv = TimeSeriesSplit(n_splits=5)

# Use the last fold
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

print("=" * 50)
print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))
print("=" * 50)

# -------------------------
# Convert to PyTorch Tensor
# -------------------------
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# -------------------------
# Create DataLoader
# -------------------------
batch_size = 32

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train Loader Batches :", len(train_loader))
print("Test Loader Batches  :", len(test_loader))

Training Samples : 2188
Testing Samples  : 437
Train Loader Batches : 69
Test Loader Batches  : 14


In [14]:
# ==========================================================
# PART 6: TRANSFORMER MODEL
# ==========================================================

import math
import torch
import torch.nn as nn

# -------------------------
# Positional Encoding
# -------------------------
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# -------------------------
# Transformer Model
# -------------------------
class CommodityTransformer(nn.Module):

    def __init__(
        self,
        input_size,
        d_model=128,
        nhead=8,
        num_layers=2,
        dropout=0.3
    ):

        super().__init__()

        self.embedding = nn.Linear(input_size, d_model)

        self.position = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)

        # Predict ONE value (next day's Gold price)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):

        x = self.embedding(x)

        x = self.position(x)

        x = self.transformer(x)

        # Last time step
        x = x[:, -1, :]

        x = self.dropout(x)

        return self.fc(x).squeeze(-1)


# -------------------------
# Initialize Model
# -------------------------
input_size = X_train.shape[2]

model = CommodityTransformer(
    input_size=input_size
).to(device)

print(model)

CommodityTransformer(
  (embedding): Linear(in_features=14, out_features=128, bias=True)
  (position): PositionalEncoding()
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [19]:
# ==========================================================
# PART 7: TRAIN THE TRANSFORMER
# ==========================================================

import copy
import torch.optim as optim

# -------------------------
# Loss Function
# -------------------------
criterion = nn.HuberLoss()

# -------------------------
# Optimizer
# -------------------------
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

# -------------------------
# LR Scheduler
# -------------------------
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)

# -------------------------
# Training Parameters
# -------------------------
epochs = 100

train_losses = []
val_losses = []

best_loss = float("inf")
best_model = None

patience = 10
counter = 0

# ==========================================================
# TRAINING LOOP
# ==========================================================

for epoch in range(epochs):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        output = model(X_batch)

        loss = criterion(output, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    train_losses.append(train_loss)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            output = model(X_batch)

            loss = criterion(output, y_batch)

            val_loss += loss.item()

    val_loss /= len(test_loader)

    val_losses.append(val_loss)

    scheduler.step(val_loss)

    # -------------------------
    # Save Best Model
    # -------------------------
    if val_loss < best_loss:

        best_loss = val_loss

        best_model = copy.deepcopy(model.state_dict())

        counter = 0

    else:

        counter += 1

    # -------------------------
    # Early Stopping
    # -------------------------
    if counter >= patience:

        print(f"\nEarly stopping at Epoch {epoch+1}")

        break

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Validation Loss: {val_loss:.5f}"
    )

# -------------------------
# Load Best Model
# -------------------------
model.load_state_dict(best_model)

print("\nTraining Completed!")
print(f"Best Validation Loss: {best_loss:.5f}")

Epoch 001 | Train Loss: 0.10981 | Validation Loss: 0.05368
Epoch 002 | Train Loss: 0.01140 | Validation Loss: 0.03246
Epoch 003 | Train Loss: 0.00847 | Validation Loss: 0.05241
Epoch 004 | Train Loss: 0.00461 | Validation Loss: 0.04806
Epoch 005 | Train Loss: 0.00457 | Validation Loss: 0.04960
Epoch 006 | Train Loss: 0.00412 | Validation Loss: 0.05101
Epoch 007 | Train Loss: 0.00396 | Validation Loss: 0.05094
Epoch 008 | Train Loss: 0.00391 | Validation Loss: 0.04608
Epoch 009 | Train Loss: 0.00476 | Validation Loss: 0.06266
Epoch 010 | Train Loss: 0.00362 | Validation Loss: 0.06037
Epoch 011 | Train Loss: 0.00359 | Validation Loss: 0.06045

Early stopping at Epoch 12

Training Completed!
Best Validation Loss: 0.03246
